## Spark Streaming With Kafka (Dashboard)

In this notebook we implement a real-time streaming pipeline that ingests weather and air-quality events from Apache Kafka, processes them with Spark Structured Streaming, and writes the results to PostgreSQL so they can be visualised live in Apache Superset.

The pipeline uses a micro-batch architecture that fits well with the roughly one-minute ingestion rate of the data sources:

```
Kafka  →  Spark Structured Streaming  →  foreachBatch  →  PostgreSQL  →  Superset
```

Two Kafka topics are consumed:
- **`weather-barcelona`** — current weather observations (temperature, wind speed, weather code, …)
- **`airquality-barcelona`** — air-quality monitoring station metadata and sensor readings

## Purpose
Run Spark Structured Streaming against Kafka weather and air-quality topics and publish dashboard-ready live evidence.

## Inputs
Kafka topics `weather-barcelona` and `airquality-barcelona`, with Spark runtime and optional dashboard sink configuration.

## Validation / quality checks
The notebook validates Kafka/Spark connectivity, parsed schemas, streaming query status, and dashboard sink readiness.

## Transformation / cleaning logic
Kafka values are parsed by topic-specific schemas, split into weather and air-quality streams, and the dashboard stream is written to the configured sink.

## Metadata and lineage fields
Launch evidence tracks source assets, check message/status, created time, schema version, and dashboard dependency status.

## Output assets
Streaming console evidence, dashboard sink rows, and `exploitation_analytics.streaming_dashboard_launch_log` from the companion Airflow validation DAG.

## RBAC / service user used
The notebook uses Kafka reader, Spark, PostgreSQL/dashboard, and ClickHouse consumption credentials configured through environment variables.

## Notebook-DAG alignment note
The notebook owns the live streaming dashboard logic, while `consumption_zone_streaming_dashboard_trigger` validates dependencies and records launch evidence. They are aligned as companion consumption evidence and the notebook does not import DAG functions.


## Execution Notes and Batch Logging
This notebook is intentionally long-running because Spark Structured Streaming calls `foreachBatch` once per micro-batch. The batch logs print each micro-batch id, raw row count, cleaned row count, rejected row count, and PostgreSQL sink confirmation so dashboard writes can be audited from notebook output.


**Importing Useful Libraries**

**Airflow Consumption DAG Alignment**

The notebook is the interactive, long-running Spark Structured Streaming runtime. The Airflow DAG `consumption_zone_streaming_dashboard_trigger` intentionally does not run the infinite streaming loop inside a worker; it validates the Kafka topics, Spark master, and notebook mount, then records a quick launch/health event in `exploitation_analytics.streaming_dashboard_launch_log` with `source_system`, `source_assets`, `created_at`, `schema_version`, `consumption_task`, `dashboard_name`, and `trigger_status`.

Standard libraries plus the Kafka-Python consumer for ad-hoc inspection, `boto3` for optional MinIO access, and `python-dotenv` to load credentials from a `.env` file.

In [1]:
from kafka import KafkaConsumer
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import *
import json
import boto3
import io
import os
import time

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "reader"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")

# Deserializer function
def deserialize(m):
    return json.loads(m.decode("utf-8"))

In [2]:
# Airflow DAG metadata contract for the quick dashboard launch/health event.
# The interactive notebook keeps running the actual Spark streaming query below.
DASHBOARD_LAUNCH_LOG_COLUMNS = [
    "source_system",
    "source_assets",
    "created_at",
    "schema_version",
    "consumption_task",
    "dashboard_name",
    "trigger_status",
]
DASHBOARD_LAUNCH_LOG_COLUMNS


['source_system',
 'source_assets',
 'created_at',
 'schema_version',
 'consumption_task',
 'dashboard_name',
 'trigger_status']

**Creating the Spark Session**

We connect to the existing Spark cluster and declare the required JARs:
- `spark-sql-kafka-0-10` — Structured Streaming connector for Kafka.
- `postgresql` — JDBC driver used later to write micro-batches to PostgreSQL.

> **Note:** The first cell downloads the Kafka connector JAR from Maven Central.
> This takes several minutes on the first run; subsequent runs use the cached JAR.

In [3]:
spark = (
    SparkSession.builder
    .appName("BDM_Streaming")
    .master("spark://spark-master:7077")
    .config(
        "spark.jars.packages",
        ",".join([
            # Downloads the Kafka connector on first run; cached in ~/.ivy2 afterward
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1",
            # Postgre JDBC driver
            "org.postgresql:postgresql:42.7.3"
        ])
    )
    .getOrCreate()
)

**Reading the Raw Kafka Stream**

We subscribe to both topics simultaneously using a single `readStream` source. Kafka delivers all messages as raw bytes. The `value` column will be parsed against explicit schemas in the next steps. `startingOffsets: earliest` allows the stream to read historical messages when no checkpoint exists; once a checkpoint exists, Spark resumes from committed offsets. Because the JDBC sink uses `append` mode, this should still not be described as duplicate-free.

In [4]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather-barcelona, airquality-barcelona") # Kafka Consumers
    .option("startingOffsets", "earliest")
    .load()
)

**Splitting Streams by Topic**

Kafka attaches a `topic` column to every row. We filter on it to produce two independent streaming DataFrames that can be processed with different schemas.

In [5]:
weather_stream = raw_stream.filter(col("topic") == "weather-barcelona")
air_stream     = raw_stream.filter(col("topic") == "airquality-barcelona")

**Defining and Applying Schemas**

**Weather stream** — flat JSON object with seven fields. We parse the `value` bytes, apply the schema, and flatten the result into a regular DataFrame (`weather_events`).

In [6]:
# Weather Schema
weather_schema = StructType([
    StructField("interval", IntegerType()),
    StructField("is_day", IntegerType()),
    StructField("temperature", DoubleType()),
    StructField("time", TimestampType()),
    StructField("weathercode", IntegerType()),
    StructField("winddirection", IntegerType()),
    StructField("windspeed", DoubleType()),
])

# Weather Events
weather_events = (
    weather_stream
    .select(from_json(col("value").cast("string"), weather_schema).alias("data"))
    .select("data.*")
)

**Air-quality stream** — JSON array of station objects with nested sub-documents (`country`, `owner`, `provider`, `coordinates`, `datetimeFirst`, `datetimeLast`). We:
1. Parse the array with `from_json`.
2. `explode` it so each station becomes its own row.
3. Select only the columns relevant to a dashboard (station identity, location, timestamps, and sensors), discarding `licenses`, `bounds`, and `instruments`.

In [7]:
# Air Schema
air_schema = ArrayType(StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("locality", StringType()),
    StructField("timezone", StringType()),

    StructField("country", StructType([
        StructField("id", IntegerType()),
        StructField("code", StringType()),
        StructField("name", StringType())
    ])),

    StructField("owner", StructType([
        StructField("id", IntegerType()),
        StructField("name", StringType())
    ])),

    StructField("provider", StructType([
        StructField("id", IntegerType()),
        StructField("name", StringType())
    ])),

    StructField("isMobile", BooleanType()),
    StructField("isMonitor", BooleanType()),

    StructField("coordinates", StructType([
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType())
    ])),

    StructField("instruments", ArrayType(StringType())),
    StructField("sensors", ArrayType(StringType())),
    StructField("licenses", ArrayType(StringType())),

    StructField("bounds", ArrayType(DoubleType())),

    StructField("datetimeFirst", StructType([
        StructField("utc", StringType()),
        StructField("local", StringType())
    ])),

    StructField("datetimeLast", StructType([
        StructField("utc", StringType()),
        StructField("local", StringType())
    ]))
]))

# Apply Air Schema
parsed = air_stream.select(
    from_json(col("value").cast("string"), air_schema).alias("data")
)

# Explode rows
air_rows = parsed.select(
    explode(col("data")).alias("station")
)

# Air Events (We are not interested in all attributes)
air_events = air_rows.select(
    col("station.id").alias("station_id"),
    col("station.name").alias("station_name"),
    col("station.locality"),
    col("station.timezone"),

    col("station.isMobile"),
    col("station.isMonitor"),

    col("station.coordinates.latitude").alias("latitude"),
    col("station.coordinates.longitude").alias("longitude"),

    col("station.country.name").alias("country"),
    col("station.provider.name").alias("provider"),
    col("station.owner.name").alias("owner"),

    col("station.datetimeFirst.local").alias("datetimeFirst"),
    col("station.datetimeLast.local").alias("datetimeLast"),

    col("station.sensors")
    #col("station.licenses")
    #col("station.bounds")
    #col("station.instruments")
)

**Console Sink — Observing Live Events**

Before writing to PostgreSQL it is useful to verify that the schemas are parsed correctly by printing micro-batches to the container's stdout. Because the output goes to the JupyterLab process log rather than the notebook, you need to run the following command in a terminal to see it:

```bash
docker compose logs jupyter -f
```

The queries are started below and must be explicitly stopped with `.stop()` before launching the PostgreSQL sink.

In [8]:
# Write to console — stop query_weather.stop()
query_weather = (
    weather_events
    .writeStream
    .format("console")
    .option("truncate", False)
    .option("numRows", 10)
    .start()
)

In [9]:
# Write to console — stop with query_air.stop()
query_air = (
    air_events
    .writeStream
    .format("console")
    .option("truncate", False)
    .option("numRows", 10)
    .start()
)

Stop both console queries once the output looks correct.

In [10]:
query_weather.stop()
query_air.stop()

### Real-time Dashboard Construction

In the end, we decided to use `weather-barcelona` only for the live dashboard because its flat schema maps directly to a relational table without any array expansion. The sink function `write_to_postgres` is called once per micro-batch by `foreachBatch`. It appends the batch DataFrame to the `weather_events` table in the `analytics` PostgreSQL database that Superset reads from. A checkpoint directory is required by Spark Structured Streaming to track committed Kafka offsets across restarts and reduce reprocessing risk; because the JDBC sink appends rows, this should not be described as exactly-once or duplicate-free.

In [11]:
def write_to_postgres(batch_df, batch_id):

    raw_count = batch_df.count()
    print(f"[micro-batch {batch_id}] received rows={raw_count}")

    # --- Preprocessing / Data validation ---
    cleaned_df = batch_df.filter(
        (col("temperature").between(-50, 60)) &
        (col("windspeed").between(0, 150)) &
        (col("winddirection").between(0, 360)) &
        (col("interval") > 0) &
        (col("time").isNotNull())
    )
    cleaned_count = cleaned_df.count()
    print(f"[micro-batch {batch_id}] cleaned rows={cleaned_count}; rejected={raw_count - cleaned_count}")

    if cleaned_count == 0:
        print(f"[micro-batch {batch_id}] skipped PostgreSQL write because the cleaned batch is empty")
        return

    # --- Write to PostgreSQL ---
    cleaned_df.write \
        .format("jdbc") \
        .mode("append") \
        .option("url", "jdbc:postgresql://postgres-analytics:5432/analytics") \
        .option("driver", "org.postgresql.Driver") \
        .option("dbtable", "weather_events") \
        .option("user", "superset") \
        .option("password", "superset") \
        .save()
    print(f"[micro-batch {batch_id}] wrote {cleaned_count} rows to PostgreSQL table weather_events")
    
# Write a real-time table
query = (
    weather_events.writeStream
    .foreachBatch(write_to_postgres)
    .option("checkpointLocation", "/tmp/checkpoints/weather") # Save the temporal file in the jupyter container
    .outputMode("append")
    .start()
)

**Monitoring the streaming query**

The cells below allow us to inspect whether the query is still running (`isActive`), print the schema to confirm the structure sent to PostgreSQL, and check the current query status (e.g. waiting for new Kafka messages vs. actively processing a batch).

In [12]:
query.isActive

True

In [13]:
weather_events.printSchema()

root
 |-- interval: integer (nullable = true)
 |-- is_day: integer (nullable = true)
 |-- temperature: double (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- weathercode: integer (nullable = true)
 |-- winddirection: integer (nullable = true)
 |-- windspeed: double (nullable = true)



In [20]:
query.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

[micro-batch 16] received rows=1
[micro-batch 16] cleaned rows=1; rejected=0
[micro-batch 16] wrote 1 rows to PostgreSQL table weather_events
[micro-batch 17] received rows=0
[micro-batch 17] cleaned rows=0; rejected=0
[micro-batch 17] skipped PostgreSQL write because the cleaned batch is empty
[micro-batch 18] received rows=1
[micro-batch 18] cleaned rows=1; rejected=0
[micro-batch 18] wrote 1 rows to PostgreSQL table weather_events
[micro-batch 19] received rows=0
[micro-batch 19] cleaned rows=0; rejected=0
[micro-batch 19] skipped PostgreSQL write because the cleaned batch is empty
[micro-batch 20] received rows=1
[micro-batch 20] cleaned rows=1; rejected=0
[micro-batch 20] wrote 1 rows to PostgreSQL table weather_events
[micro-batch 21] received rows=0
[micro-batch 21] cleaned rows=0; rejected=0
[micro-batch 21] skipped PostgreSQL write because the cleaned batch is empty
[micro-batch 22] received rows=1
[micro-batch 22] cleaned rows=1; rejected=0
[micro-batch 22] wrote 1 rows to Po

Stop the streaming query when done. The checkpoint will be preserved so the next run resumes from the last committed Kafka offset.

In [21]:
query.stop()